# Stage 1B: U-Net control

Fair control for the spectral-loss comparison: the U-Net + MSE learning rate is tuned first, and
the H1-type spectral loss ($\lambda=0.5$, $|k|^2$ weights) is then trained at the same learning rate.
The learning rate is picked on validation loss only; the test set is not used for any choice.

| setting | value |
|---|---|
| model | U-Net (base 32), 2,140,065 params |
| lr sweep (MSE) | $10^{-3}$, $3\cdot10^{-4}$, $10^{-4}$ |
| selection | lowest best validation MSE |
| spectral run | SpectralLoss(0.5, "k2") at the selected lr, best epoch by validation MSE |
| epochs / batch / seed | 20 / 32 / 0 |

The ring-weighted variant is not rerun (it had a $k=0$ ring bug in v1).

## Setup

In [ ]:
import sys, glob, os, importlib.util
if importlib.util.find_spec("neuralop") is None:
    !pip install -q neuraloperator
# project code (nsbench, optim, solver) comes from the private ns-surrogate-code dataset
code_dir = next((os.path.dirname(os.path.dirname(p)) for p in
                 glob.glob("/kaggle/input/**/nsbench/__init__.py", recursive=True)), os.path.abspath(".."))
sys.path.insert(0, code_dir)
import torch, numpy as np, pandas as pd
import matplotlib.pyplot as plt
import nsbench
from nsbench import Trainer, save_csv, save_fig, save_predictions, run_record
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pd.DataFrame([{"python": sys.version.split()[0], "torch": torch.__version__,
               "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
               "code": code_dir}])

## Data

In [ ]:
D = nsbench.load()
pd.DataFrame([{"data": D.data_dir, "train batches": len(D.train_loader), "val batches": len(D.val_loader),
               "test samples": len(D.x_test), "y_mean": D.y_mean.item(), "y_std": D.y_std.item()}])

## Learning-rate sweep (MSE)

In [ ]:
LRS, EPOCHS, SEED = [1e-3, 3e-4, 1e-4], 20, 0
rows, hists, trainers = [], [], {}
for lr in LRS:
    tr = Trainer(nsbench.UNet, lambda p, lr=lr: torch.optim.Adam(p, lr=lr),
                 D.train_loader, D.val_loader, D.test_loader, D.y_mean, D.y_std, device,
                 monitor="loss", seed=SEED, label=f"U-Net MSE lr {lr:g}").fit(EPOCHS)
    rows.append(run_record(tr.summary(), run="U-Net", optimizer="Adam", lr=lr, seed=SEED,
                           epochs=EPOCHS, batch=32, loss_fn="MSE", monitor="loss"))
    hists.append(tr.history().assign(run=f"MSE lr {lr:g}"))
    trainers[lr] = tr
sweep = pd.concat(rows, ignore_index=True)
save_csv(sweep, "stage1_unet_lr_sweep")
sweep[["label", "best_epoch", "best_val_loss", "test_rel_L2", "sec_per_epoch"]]

## Selected learning rate

In [ ]:
best_lr = float(sweep.loc[sweep.best_val_loss.idxmin(), "lr"])
pd.DataFrame([{"selected lr": best_lr, "criterion": "lowest best_val_loss (validation MSE)"}])

## Spectral loss at the selected learning rate

In [ ]:
tr_k2 = Trainer(nsbench.UNet, lambda p: torch.optim.Adam(p, lr=best_lr),
                D.train_loader, D.val_loader, D.test_loader, D.y_mean, D.y_std, device,
                loss_fn=nsbench.SpectralLoss(0.5, "k2"), monitor="mse_norm", seed=SEED,
                label=f"U-Net spectral k2 lr {best_lr:g}").fit(EPOCHS)
k2 = run_record(tr_k2.summary(), run="U-Net", optimizer="Adam", lr=best_lr, seed=SEED, epochs=EPOCHS,
                batch=32, loss_fn="SpectralLoss(0.5,k2)", monitor="mse_norm")
hists.append(tr_k2.history().assign(run=f"spectral k2 lr {best_lr:g}"))
control = pd.concat([sweep[sweep.lr == best_lr], k2], ignore_index=True)
save_csv(control, "stage1_unet_control")
save_csv(pd.concat(hists, ignore_index=True), "stage1_unet_history")
control[["label", "best_epoch", "test_rel_L2", "test_mse_norm", "sec_per_epoch"]]

## Error by frequency band (test set, raw units)

In [ ]:
from nsbench.analysis import band_errors
raw = lambda t: t * D.y_std + D.y_mean
y_raw = raw(D.y_test.squeeze(1))
preds = {"U-Net MSE": raw(save_predictions(trainers[best_lr], D.x_test, f"unet_mse_lr{best_lr:g}")),
         "U-Net spectral k2": raw(save_predictions(tr_k2, D.x_test, f"unet_k2_lr{best_lr:g}"))}
bands = band_errors(preds, y_raw)
save_csv(bands, "stage1_unet_band_errors")
bands

## Validation curves

In [ ]:
H = pd.concat(hists, ignore_index=True)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for name, h in H.groupby("run", sort=False):
    ax[0].plot(h.epoch, h.val_mse_norm, label=name)
    ax[1].plot(h.epoch, h.val_rel_L2, label=name)
ax[0].set(xlabel="epoch", ylabel="validation MSE (standardized)", yscale="log")
ax[1].set(xlabel="epoch", ylabel="validation relative $L^2$", yscale="log")
ax[0].legend(); fig.suptitle("U-Net: lr sweep and spectral loss"); fig.tight_layout()
save_fig("stage1_unet_curves", fig)